# Malaysia Tax Laws — Vector Ingestion Pipeline

Loads PDFs from `data/`, enriches them with structured metadata, splits into chunks, and ingests into Qdrant.

**Run cells in order.** Ingestion is idempotent — files already in the store are skipped based on their content hash.

## 1. Imports & Configuration

In [22]:
import os
import hashlib
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
load_dotenv(Path("../../.env"))

from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue, PayloadSchemaType

DATA_PATH = Path("../../data/")
COLLECTION_NAME = "malaysia-tax-laws"
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

# Chunk size tuned for legal text: longer sentences, section-based structure
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 250
BATCH_SIZE = 100

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSIONS = 1536

## 2. Document Registry

Each document is explicitly registered with structured metadata. This drives:
- **Filtering** at query time — e.g. search only Public Rulings, or only documents from 2020+
- **Source attribution** in the UI — title and reference number shown alongside answers
- **Authority ranking** — Act (1) > Public Ruling (2) > Guidelines/Announcements (3)
- **Deduplication** — `source_hash` prevents re-ingesting unchanged files

Add new documents here before running ingestion.

In [23]:
DOCUMENT_REGISTRY: dict[str, dict] = {
    "income-tax-act-1967-act-53.pdf": {
        "doc_type": "primary_legislation",
        "doc_subtype": "act",
        "title": "Income Tax Act 1967",
        "reference": "Act 53",
        "issuing_body": "Parliament of Malaysia",
        "year": 1967,
        "topics": ["income tax", "general provisions", "assessment", "appeals"],
        "authority_level": 1,
    },
    "Residence_Status_PR_11_2017.pdf": {
        "doc_type": "public_ruling",
        "doc_subtype": "residence",
        "title": "Residence Status of Individuals",
        "reference": "PR 11/2017",
        "issuing_body": "LHDN",
        "year": 2017,
        "topics": ["residence status", "tax residency", "183 days", "individual"],
        "authority_level": 2,
    },
    "Residence_tax_pr-5_2022.pdf": {
        "doc_type": "public_ruling",
        "doc_subtype": "residence",
        "title": "Residence Status of Individuals (Updated)",
        "reference": "PR 5/2022",
        "issuing_body": "LHDN",
        "year": 2022,
        "topics": ["residence status", "tax residency", "individual"],
        "authority_level": 2,
    },
    "Tax_releif_PR2_2012.pdf": {
        "doc_type": "public_ruling",
        "doc_subtype": "tax_relief",
        "title": "Tax Relief for Resident Individual",
        "reference": "PR 2/2012",
        "issuing_body": "LHDN",
        "year": 2012,
        "topics": ["tax relief", "deductions", "resident individual"],
        "authority_level": 2,
    },
    "Tax_treatment_PR8_2011.pdf": {
        "doc_type": "public_ruling",
        "doc_subtype": "employment_income",
        "title": "Tax Treatment of Employee Benefits",
        "reference": "PR 8/2011",
        "issuing_body": "LHDN",
        "year": 2011,
        "topics": ["employment income", "benefits in kind", "perquisites"],
        "authority_level": 2,
    },
    "Abroad_income_20240620-guidelines-tax-treatment-in-relation-to-income-received-from-abroad-amendment-june-2024.pdf": {
        "doc_type": "guidelines",
        "doc_subtype": "foreign_income",
        "title": "Tax Treatment: Income Received from Abroad",
        "reference": "Guidelines June 2024",
        "issuing_body": "LHDN",
        "year": 2024,
        "topics": ["foreign income", "FSI", "remittance", "overseas"],
        "authority_level": 3,
    },
    "tax_incentives_orgs_technical_announcements_250102_1_2.pdf": {
        "doc_type": "technical_announcement",
        "doc_subtype": "tax_incentives",
        "title": "Tax Incentives for Organisations",
        "reference": "Technical Announcement Jan 2025",
        "issuing_body": "LHDN",
        "year": 2025,
        "topics": ["tax incentives", "organisations", "exemptions"],
        "authority_level": 3,
    },
}

print(f"Registry contains {len(DOCUMENT_REGISTRY)} documents")

Registry contains 7 documents


## 3. Client & Splitter Setup

Creates the Qdrant collection if it doesn't exist. **Cosine similarity** is the right distance metric for semantic search — it measures the angle between vectors rather than magnitude, which is what you want when comparing meaning.

In [24]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBEDDING_DIMENSIONS, distance=Distance.COSINE),
    )
    print(f"Created collection: {COLLECTION_NAME}")
else:
    print(f"Collection already exists: {COLLECTION_NAME}")

# Payload index required for filtering on this field during deduplication checks.
# Safe to call repeatedly — Qdrant ignores it if the index already exists.
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="metadata.source_hash",
    field_schema=PayloadSchemaType.KEYWORD,
)
print("Payload index ready: metadata.source_hash")

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

# Prefer paragraph and section breaks to keep legal clauses intact
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n\n", "\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

print("Setup complete.")

Collection already exists: malaysia-tax-laws
Payload index ready: metadata.source_hash
Setup complete.


## 4. Helper Functions

In [25]:
def get_file_hash(path: Path) -> str:
    """MD5 fingerprint of file contents — used to detect unchanged files."""
    return hashlib.md5(path.read_bytes()).hexdigest()


def is_already_ingested(client: QdrantClient, collection_name: str, file_hash: str) -> bool:
    """Returns True if any chunk with this source_hash already exists in the collection."""
    results, _ = client.scroll(
        collection_name=collection_name,
        scroll_filter=Filter(
            must=[FieldCondition(key="metadata.source_hash", match=MatchValue(value=file_hash))]
        ),
        limit=1,
    )
    return len(results) > 0

## 5. Ingest Documents

For each PDF in `data/`:
1. Checks the registry — unregistered files are skipped with a warning
2. Computes a content hash and checks if it's already in the store — skips if so
3. Loads pages, merges registry metadata onto every chunk
4. Ingests in batches of `BATCH_SIZE` to avoid timeouts

In [26]:
pdf_files = sorted(DATA_PATH.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files\n")

all_chunks = []

for pdf_path in pdf_files:
    registry_entry = DOCUMENT_REGISTRY.get(pdf_path.name)
    if not registry_entry:
        print(f"  WARN  {pdf_path.name} — not in registry, skipping")
        continue

    file_hash = get_file_hash(pdf_path)

    if is_already_ingested(client, COLLECTION_NAME, file_hash):
        print(f"  SKIP  {pdf_path.name}")
        continue

    docs = PyPDFLoader(str(pdf_path)).load()

    for doc in docs:
        doc.metadata.update({
            **registry_entry,
            "source_file": pdf_path.name,
            "source_hash": file_hash,
            "page": doc.metadata.get("page", 0),
            "ingested_at": datetime.now(timezone.utc).isoformat(),
        })

    chunks = splitter.split_documents(docs)
    all_chunks.extend(chunks)
    print(f"  LOAD  {pdf_path.name} ({len(chunks)} chunks)")

print(f"\nTotal new chunks to ingest: {len(all_chunks)}")

if all_chunks:
    total_batches = (len(all_chunks) + BATCH_SIZE - 1) // BATCH_SIZE
    for i in range(0, len(all_chunks), BATCH_SIZE):
        batch = all_chunks[i : i + BATCH_SIZE]
        vector_store.add_documents(batch)
        print(f"  Batch {i // BATCH_SIZE + 1}/{total_batches} ingested")
    print(f"\nDone. {len(all_chunks)} chunks added to '{COLLECTION_NAME}'.")
else:
    print("Nothing to ingest — all files already in the store.")

Found 7 PDF files

  LOAD  Abroad_income_20240620-guidelines-tax-treatment-in-relation-to-income-received-from-abroad-amendment-june-2024.pdf (65 chunks)
  LOAD  Residence_Status_PR_11_2017.pdf (49 chunks)
  LOAD  Residence_tax_pr-5_2022.pdf (89 chunks)
  LOAD  Tax_releif_PR2_2012.pdf (39 chunks)
  LOAD  Tax_treatment_PR8_2011.pdf (34 chunks)
  LOAD  income-tax-act-1967-act-53.pdf (1374 chunks)
  LOAD  tax_incentives_orgs_technical_announcements_250102_1_2.pdf (17 chunks)

Total new chunks to ingest: 1667
  Batch 1/17 ingested
  Batch 2/17 ingested
  Batch 3/17 ingested
  Batch 4/17 ingested
  Batch 5/17 ingested
  Batch 6/17 ingested
  Batch 7/17 ingested
  Batch 8/17 ingested
  Batch 9/17 ingested
  Batch 10/17 ingested
  Batch 11/17 ingested
  Batch 12/17 ingested
  Batch 13/17 ingested
  Batch 14/17 ingested
  Batch 15/17 ingested
  Batch 16/17 ingested
  Batch 17/17 ingested

Done. 1667 chunks added to 'malaysia-tax-laws'.
